In [ ]:
#!pip install -U  byaldi pdf2image qwen-vl-utils transformers ipywidgets jinja2 einops seaborn
# Tested with byaldi==0.0.4, pdf2image==1.17.0, qwen-vl-utils==0.0.8, transformers==4.45.0
!pip install --user --upgrade \
    "numpy<2.0.0"
!pip install colpali-engine==0.3.4 transformers==4.46.1 huggingface-hub==0.24.0 tokenizers==0.20.3 pdf2image qwen-vl-utils matplotlib pillow tqdm seaborn einops jinja2 ipywidgets peft accelerate bitsandbytes
!apt-get install -y poppler-utils

In [2]:
import os

# 1. 打印 input 目录下的所有文件，确认你的数据集文件夹名称
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/blackmanba23/toyotapdf/rav_4.pdf


In [3]:
# !sudo apt-get install -y poppler-utils

In [ ]:
# colpali modules
from colpali_engine.models import ColQwen2, ColQwen2Processor
from colpali_engine.models import ColPali, ColPaliProcessor
from colpali_engine.interpretability import (
    get_similarity_maps_from_embeddings,
    plot_all_similarity_maps,
    plot_similarity_map,
)

#vison language models
from transformers import Qwen2VLForConditionalGeneration, Qwen2VLProcessor
from qwen_vl_utils import process_vision_info

import torch
from torch.utils.data import DataLoader
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

import os
from tqdm import tqdm
from PIL import Image
from pdf2image import convert_from_path
import base64
from io import BytesIO
import matplotlib.pyplot as plt
import pprint

# 使用 ColPali 和 Qdrant 构建基于视觉的检索系统

- 本notebook演示了使用 ColPali 和 Qwen2-VL 创建基于视觉的检索系统的端到端流程。在同仓库的[配套notebook]()中，我们还将介绍如何与向量存储框架 [Qdrant](https://qdrant.tech/) 通信以更高效地执行此流程。本notebook主要概述了该问题的更底层方法。

- [ColPali](https://github.com/illuin-tech/colpali) 是文档检索领域的一项最新突破，它利用视觉语言模型（VLMs）直接从文档图像创建高效的多向量嵌入。其核心思想是将每个PDF页面视为图像，并使用视觉语言模型来生成嵌入。这种方法大大消除了对复杂且脆弱的版面识别和OCR流水线的需求。实际上，它在多个基准测试中大幅超越了其他模型。作为额外的好处，我们还将可视化相似度图，以查看模型在检索过程中在token级别上关注的内容。

- 我们将把 [Qwen2-VL](https://huggingface.co/Qwen/Qwen2-VL-7B-Instruct) 集成到我们的检索系统中。这是阿里巴巴著名的视觉语言模型。我们希望能够基于检索到的图像生成详细且与上下文相关的回答。


## 从PDF创建文档嵌入

这些是我们将使用的模型，其他模型请查看 ColPali 官网。我们可以使用 "bfloat16" 来加速推理。

In [ ]:
from transformers import BitsAndBytesConfig

model_name = "vidore/colpali-v1.2"
processor = ColPaliProcessor.from_pretrained(model_name)

# 4-bit 量化
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

model = ColPali.from_pretrained(
    model_name,
    device_map="cuda:0",
    quantization_config=bnb_config,
).eval()


对于 ColPali，每个PDF页面就是一张图像，因此我们首先将所有PDF页面转换为图像，同时保留必要的元数据。

In [6]:
pdf_dir = '/kaggle/input/datasets/blackmanba23/toyotapdf/'

In [ ]:
def convert_pdf_to_images(pdf_dir):
    """
    Converts all PDFs in a directory to images.

    Args:
        pdf_dir (str): Path to the directory containing PDFs.

    Returns:
        dict: A dictionary where keys are file names (without extension), 
              and values are lists of images (one list per PDF).
    """
    pdf_list = [pdf for pdf in os.listdir(pdf_dir) if pdf.endswith(".pdf")]
    all_images = {}

    for pdf_file in pdf_list:
        pdf_path = os.path.join(pdf_dir, pdf_file)
        pdf_images = convert_from_path(pdf_path)
        all_images[pdf_file] = pdf_images  # Use file name as key
    
    return all_images


all_images = convert_pdf_to_images(pdf_dir)

In [8]:
print(all_images.keys())

dict_keys(['rav_4.pdf'])


我们的示例PDF是一本丰田RAV4的口袋用户手册，但您可以在下面的文件夹中放入任意数量的不同PDF文件。
从示例中可以看出，这个PDF对于任何基于OCR的检索模型来说都是极其困难的。

In [ ]:
#lets see one sample
fig, ax = plt.subplots(figsize=(15,15))
ax.imshow(all_images['rav_4.pdf'][7])
ax.grid(False)
ax.set_xticks([])
ax.set_yticks([])
plt.show()

In [ ]:
def create_document_embeddings(pdf_dir, model, processor, batch_size=2):
    """
    Converts all PDFs in a directory into embeddings with metadata.

    Args:
        pdf_dir (str): Directory containing PDF files.
        model: Pre-trained model for generating embeddings.
        processor: Preprocessor for the model (e.g., to process images).
        batch_size (int): Batch size for inference.

    Returns:
        list: A list of dictionaries, where each dictionary contains:
            - "embedding": The embedding tensor.
            - "doc_id": The document ID (int).
            - "page_id": The page index within the document.
            - "file_name": The name of the source PDF file.
    """
    all_images = convert_pdf_to_images(pdf_dir)
    all_embeddings_with_metadata = []

    for doc_id, (file_name, pdf_images) in enumerate(all_images.items()):
        dataloader = DataLoader(
            dataset=pdf_images,
            batch_size=batch_size,
            shuffle=False,
            collate_fn=lambda x: processor.process_images(x),
        )

        page_counter = 0
        for batch in tqdm(dataloader, desc=f"Processing {file_name}"):
            with torch.no_grad():
                batch = {k: v.to(model.device) for k, v in batch.items()}
                batch_embeddings = model(**batch)
                batch_embeddings = list(torch.unbind(batch_embeddings.to("cpu")))

                for embedding in batch_embeddings:
                    all_embeddings_with_metadata.append({
                        "embedding": embedding,
                        "doc_id": doc_id,
                        "page_id": page_counter,
                        "file_name": file_name,  # Correctly use the file name
                    })
                    page_counter += 1

    return all_embeddings_with_metadata

In [11]:
ds = create_document_embeddings(pdf_dir, model, processor, batch_size=2)
print(f"Generated embeddings for {len(ds)} pages.")

Processing rav_4.pdf: 100%|██████████| 7/7 [00:08<00:00,  1.16s/it]

Generated embeddings for 13 pages.


In [ ]:
# here is the first page
ds[0]

## 基于用户查询的检索

现在我们已经准备好根据用户查询检索页面。请注意，ColPali 成功识别了正确的页面。

In [ ]:
def get_results(query, processor, model, ds, all_images, top_k=5):
    """
    Retrieves top-k relevant images for a given query.

    Args:
        query (str): User query as a string.
        processor: Processor for pre-processing the query.
        model: Model to generate embeddings for the query.
        ds (list): List of dictionaries with "embedding", "doc_id", "page_id", and "file_name".
        all_images (dict): Dictionary of images per document.
        top_k (int): Number of top results to retrieve.

    Returns:
        list: A list of dictionaries, where each dictionary contains:
            - "doc_id": Document ID.
            - "page_id": Page ID.
            - "file_name": Name of the source PDF file.
            - "image": The retrieved image (PIL.Image.Image).
            - "score": Similarity score for the image.
    """
    # Process the query and move to model's device
    batch_queries = processor.process_queries([query]).to(model.device)

    # Forward pass to get query embeddings
    with torch.no_grad():
        query_embeddings = model(**batch_queries)

    # Extract embeddings from ds for scoring
    document_embeddings = torch.stack([entry["embedding"] for entry in ds])
    
    # Compute similarity scores
    scores = processor.score_multi_vector(query_embeddings, document_embeddings)
    score_values = scores[0].tolist()  # Extract similarity scores as a list

    # Get top-k indices of the most relevant embeddings
    top_indices = scores[0].topk(top_k).indices.tolist()

    # Retrieve corresponding images and metadata
    retrieved_results = []
    for idx in top_indices:
        entry = ds[idx]
        doc_id = entry["doc_id"]
        page_id = entry["page_id"]
        file_name = entry["file_name"]
        image = all_images[file_name][page_id]  # Correct lookup using file_name

        # Add score to each result
        retrieved_results.append({
            "doc_id": doc_id,
            "page_id": page_id,
            "file_name": file_name,
            "image": image,
            "score": score_values[idx],  # Add similarity score
        })

    return retrieved_results

In [14]:
query = "What is the purpose of 'AUTO LSD' switch?"
retrieved_results = get_results(
    query = query,
    processor=processor,
    model=model,
    ds=ds,
    all_images=all_images,
    top_k=2,
)

retrieved_results

[{'doc_id': 0,
  'page_id': 11,
  'file_name': 'rav_4.pdf',
  'image': <PIL.PpmImagePlugin.PpmImageFile image mode=RGB size=1700x2200>,
  'score': 18.53125},
 {'doc_id': 0,
  'page_id': 4,
  'file_name': 'rav_4.pdf',
  'image': <PIL.PpmImagePlugin.PpmImageFile image mode=RGB size=1700x2200>,
  'score': 17.625}]

In [ ]:
def display_top_two_results(retrieved_results):
    """
    Displays the top two retrieved images in a 1x2 canvas using Matplotlib.

    Args:
        retrieved_results (list): A list of dictionaries with metadata and images.

    Returns:
        None
    """
    # Take only the top two results
    top_two_results = retrieved_results[:2]

    # Create a 1x2 canvas
    fig, axes = plt.subplots(1, 2, figsize=(16, 16))

    for i, result in enumerate(top_two_results):
        file_name = result["file_name"]
        page_id = result["page_id"]
        score = result["score"]
        image = result["image"]

        axes[i].imshow(image)
        axes[i].axis('off')
        axes[i].set_title(f"File: {file_name}\nPage: {page_id+1}\nScore: {score:.4f}")

    plt.tight_layout()
    plt.show()
display_top_two_results(retrieved_results)

## 可解释的检索

- 仅凭 ColPali 的构造，它就允许我们通过相似度矩阵来解释查询中每个token的关注位置。在本节结束时，我们将看到 ColPali 以可解释的方式识别图像中的相关部分。这使我们能够在检索效果不佳时进一步优化模型。
- 注意 "LSD" 这个词在检索到的图像中是如何被完美识别的，包括其周围的token。请注意，这个过程可以从更详细的查询中受益。

In [ ]:
# this is the top ranking retrived image
image = retrieved_results[0]['image']

In [ ]:
def get_query_tokens(query, processor):
    # process query
    batch_queries = processor.process_queries([query]).to(device)
    
    # get the exact token IDs after model-spesific tokenization
    query_content = processor.decode(batch_queries.input_ids[0]).replace(processor.tokenizer.pad_token, "")
    query_content = query_content.replace(processor.query_augmentation_token, "").strip()
    query_tokens = processor.tokenizer.tokenize(query_content)

    return query_tokens

# now based on token ID, we can inspect the simuilari scores
query_tokens = get_query_tokens(query,processor)
pprint.pprint({idx: val for idx, val in enumerate(query_tokens)})

In [ ]:
def get_similarity_map(query, image, model, processor):
    batch_images = processor.process_images([image]).to(device)   # [B, C, 448, 448]
    batch_queries = processor.process_queries([query]).to(device) # [B, query_lenght]
    
    # Forward passes
    with torch.no_grad():
        image_embeddings = model.forward(**batch_images)          # [B, 1030, 128]
        query_embeddings = model.forward(**batch_queries)         # [B, query_lenght, 128]
    
    # Get the number of image patches
    n_patches = processor.get_n_patches(image_size=image.size, patch_size=model.patch_size) # [32, 32]

    
    # Get the tensor mask to filter out the embeddings that are not related to the image
    image_mask = processor.get_image_mask(batch_images)

    # Generate the similarity maps
    batched_similarity_maps = get_similarity_maps_from_embeddings(
        image_embeddings=image_embeddings,
        query_embeddings=query_embeddings,
        n_patches=n_patches,
        image_mask=image_mask,
    )

    # Get the similarity map for the input image. Note that each token is compared to all image patches
    similarity_maps = batched_similarity_maps[0]  # (query_length, n_patches_x, n_patches_y)

    return similarity_maps
    
similarity_maps =  get_similarity_map(query, image, model, processor)

In [ ]:
def display_combined_similarity_map(similarity_maps, image, token_indices):
    """
    Display the combined similarity map for multiple tokens.

    Args:
        similarity_maps: The similarity maps tensor (query_length, n_patches_x, n_patches_y).
        image: The original image.
        token_indices: A list of token indices to combine for the similarity map.
    """
    selected_tokens = [query_tokens[idx] for idx in token_indices]
    #print(f"Selected tokens: {selected_tokens}")
    
    # Combine the similarity maps for the selected tokens by averaging the maps across selected tokens
    combined_similarity_map = similarity_maps[token_indices].mean(dim=0)  
    
    # Plot the combined similarity map
    fig, ax = plot_similarity_map(
        image=image,
        similarity_map=combined_similarity_map,
        figsize=(12, 12),
        show_colorbar=False,
    )
    
    # note that this is the max of average similarity scores
    max_sim_score = combined_similarity_map.max().item()
    ax.set_title(f"Tokens: {selected_tokens}. MaxSim score: {max_sim_score:.2f}", fontsize=14)
    
    plt.show()

# Map for token "LSD"
display_combined_similarity_map(similarity_maps, image, token_indices=[10])

In [ ]:
# AUTO + LSD + switch
display_combined_similarity_map(similarity_maps, image, token_indices=[9,10,12])

# 最后阶段：生成

现在我们已经成功从PDF文件中检索到了相关图像，我们可以将这些检索到的信息输入到视觉语言模型中。我们可以看到，到目前为止讨论的流水线基于 ColPali 检索到的图像，完美地为我们的查询提供了答案。

In [ ]:
# 7B model does not fit into my GPU
import torch
import gc

# 强行触发 Python 的垃圾回收机制
gc.collect()
# 强行让 PyTorch 释放所有未使用的缓存显存
torch.cuda.empty_cache()
# Qwen2VL 4-bit 量化
bnb_config_vl = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

vl_model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    device_map="cuda:1",
    quantization_config=bnb_config_vl,
).eval()

In [ ]:
# optimize this to fit more images to memory
min_pixels = 224 * 224
max_pixels = 512 * 512
vl_model_processor = Qwen2VLProcessor.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct", min_pixels=min_pixels, max_pixels=max_pixels
)

In [ ]:
def generate_anwers(vl_model, vl_model_processor, query, grouped_images, max_new_tokens=100):
    """
    在 cuda:1 上生成回答
    """
    chat_template = [
        {
            "role": "user",
            "content": [{"type": "image", "image": image} for image in grouped_images]
            + [{"type": "text", "text": query}],
        }
    ]
    text = vl_model_processor.apply_chat_template(chat_template, tokenize=False, 
                                              add_generation_prompt=True)
    image_inputs, _ = process_vision_info(chat_template)
    inputs = vl_model_processor(
        text=[text],
        images=image_inputs,
        padding=True,
        return_tensors="pt",
    )
    # 将输入移动到 cuda:1
    inputs = inputs.to("cuda:1")
    
    print("[cuda:1] Generating answer...")
    generated_ids = vl_model.generate(**inputs, max_new_tokens=max_new_tokens)
    generated_ids_trimmed = [out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
    output_text = vl_model_processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    return output_text[0]

In [ ]:
# ==================== 跨卡通信：cuda:0 → cuda:1 ====================
print("=" * 60)
print("Cross-GPU Communication: cuda:0 → cuda:1")
print("=" * 60)

# 从检索结果中提取图像
grouped_images = [result["image"] for result in retrieved_results]
print(f"[cuda:0] Retrieved {len(grouped_images)} images")

# 图像以 PIL 格式传递
print(f"[Transfer] Sending images from cuda:0 to cuda:1")

# 在 cuda:1 上生成回答
generated_response = generate_anwers(vl_model, vl_model_processor, query, grouped_images, max_new_tokens=100)

print("\n" + "=" * 60)
print("Generated Response:")
print("=" * 60)
print(generated_response)

# 完整流水线

In [ ]:
# ==================== Full Pipeline: 双 GPU 部署 ====================
if __name__ == "__main__":
    # ==================== 清空显存 ====================
    print("Clearing GPU memory...")
    torch.cuda.empty_cache()
    import gc
    gc.collect()
    print(f"GPU 0 memory: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB allocated")
    print(f"GPU 1 memory: {torch.cuda.memory_allocated(1)/1024**3:.2f} GB allocated")
    
    # ==================== 配置 ====================
    pdf_dir = '/kaggle/input/datasets/blackmanba23/toyotapdf/'
    query = "What is the purpose of 'AUTO LSD' switch?"
    colpali_model_name = "vidore/colpali-v1.2"
    vl_model_name = "Qwen/Qwen2-VL-2B-Instruct"
    top_k = 2
    max_new_tokens = 100
    
    print("=" * 70)
    print("Dual-GPU RAG Pipeline")
    print("cuda:0: ColPali (Retrieval)")
    print("cuda:1: Qwen2VL (Generation)")
    print("=" * 70)
    
    # ==================== 4-bit 量化配置 ====================
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    
    # ==================== Step 1: 加载 ColPali (cuda:0) ====================
    print("\n[Step 1] Loading ColPali on cuda:0...")
    colpali_processor = ColPaliProcessor.from_pretrained(colpali_model_name)
    colpali_model = ColPali.from_pretrained(
        colpali_model_name,
        device_map="cuda:0",
        quantization_config=bnb_config,
    ).eval()
    print(f"[OK] ColPali loaded on {colpali_model.device}")
    print(f"[Memory] GPU 0: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB")
    
    # ==================== Step 2: 加载 Qwen2VL (cuda:1) ====================
    print("\n[Step 2] Loading Qwen2VL on cuda:1...")
    qwen_model = Qwen2VLForConditionalGeneration.from_pretrained(
        vl_model_name,
        device_map="cuda:1",
        quantization_config=bnb_config,
    ).eval()
    min_pixels = 224 * 224
    max_pixels = 512 * 512
    qwen_processor = Qwen2VLProcessor.from_pretrained(
        vl_model_name, min_pixels=min_pixels, max_pixels=max_pixels
    )
    print(f"[OK] Qwen2VL loaded on {qwen_model.device}")
    print(f"[Memory] GPU 1: {torch.cuda.memory_allocated(1)/1024**3:.2f} GB")
    
    # ==================== Step 3: PDF 转图像 ====================
    print("\n[Step 3] Converting PDFs to images...")
    all_images = convert_pdf_to_images(pdf_dir)
    print(f"[OK] Converted {len(all_images)} PDFs")
    
    # ==================== Step 4: 创建嵌入 (cuda:0) ====================
    print("\n[Step 4] Creating embeddings on cuda:0...")
    ds = create_document_embeddings(pdf_dir, colpali_model, colpali_processor, batch_size=2)
    print(f"[OK] Generated embeddings for {len(ds)} pages")
    
    # ==================== Step 5: 检索 (cuda:0) ====================
    print("\n[Step 5] Retrieving on cuda:0...")
    retrieved_results = get_results(query, colpali_processor, colpali_model, ds, all_images, top_k=top_k)
    print(f"[OK] Retrieved {len(retrieved_results)} pages")
    for i, result in enumerate(retrieved_results):
        print(f"      {i+1}. Page {result['page_id']+1}, Score: {result['score']:.4f}")
    
    # ==================== Step 6: 跨卡通信 (cuda:0 → cuda:1) ====================
    print("\n[Step 6] Cross-GPU Transfer: cuda:0 → cuda:1")
    grouped_images = [result["image"] for result in retrieved_results]
    print(f"[OK] Transferred {len(grouped_images)} images")
    
    # ==================== Step 7: 生成回答 (cuda:1) ====================
    print("\n[Step 7] Generating answer on cuda:1...")
    chat_template = [
        {
            "role": "user",
            "content": [{"type": "image", "image": image} for image in grouped_images]
            + [{"type": "text", "text": query}],
        }
    ]
    text = qwen_processor.apply_chat_template(chat_template, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(chat_template)
    inputs = qwen_processor(
        text=[text], images=image_inputs, padding=True, return_tensors="pt"
    )
    inputs = inputs.to("cuda:1")
    
    generated_ids = qwen_model.generate(**inputs, max_new_tokens=max_new_tokens)
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    response = qwen_processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]
    
    print("\n" + "=" * 70)
    print("Generated Response:")
    print("=" * 70)
    print(response)
    
    # ==================== 最终显存统计 ====================
    print("\n" + "=" * 70)
    print("Final GPU Memory Usage:")
    print("=" * 70)
    print(f"GPU 0: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB allocated")
    print(f"GPU 1: {torch.cuda.memory_allocated(1)/1024**3:.2f} GB allocated")